In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

In [2]:
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch: 2.5.1+cu121
CUDA available: True
Using device: cuda


In [3]:
df = pd.read_csv(r"C:\Users\Arup Sarkar\Documents\EmoWave\data\dataemotion_data.csv")

N_MELS = 128
SR = 22050
DURATION = 3
N_FFT = 2048
HOP_LENGTH = 512
NUM_CLASSES = 8

print(f'Total clips: {len(df)}')
print(f'Classes: {df["emotion"].nunique()}')

Total clips: 1440
Classes: 8


In [8]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['emotion'])

class EmoWaveDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = row['path']

        y, sr = librosa.load(file_path, sr=SR, duration=DURATION)

        if len(y) < SR * DURATION:
            y = np.pad(y, (0, SR * DURATION - len(y)))

        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                           n_fft=N_FFT, hop_length=HOP_LENGTH)
        S_db = librosa.power_to_db(S, ref=np.max)

        S_db = (S_db - S_db.mean()) / (S_db.std() + 1e-8)

        tensor = torch.tensor(S_db, dtype=torch.float32).unsqueeze(0)

        label = torch.tensor(row['label'], dtype=torch.long)
        return tensor, label

print('Dataset class defined')

Dataset class defined


In [9]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

train_dataset = EmoWaveDataset(train_df)
test_dataset = EmoWaveDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')
print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')

# verify one batch
batch_x, batch_y = next(iter(train_loader))
print(f'Batch X shape: {batch_x.shape}')
print(f'Batch y shape: {batch_y.shape}')

Train samples: 1152
Test samples:  288
Train batches: 36
Test batches:  9
Batch X shape: torch.Size([32, 1, 128, 130])
Batch y shape: torch.Size([32])


In [10]:
dummy = torch.zeros(1, 1, 128, 130).to(device)

In [12]:
class EmoWaveNet(nn.Module):
    def __init__(self, num_classes=8):
        super(EmoWaveNet, self).__init__()

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.classifier(x)
        return x

model = EmoWaveNet(num_classes=NUM_CLASSES).to(device)

In [13]:
with torch.no_grad():
    dummy = torch.zeros(1, 1, 128, 130).to(device)
    out = model.conv_block1(dummy)
    out = model.conv_block2(out)
    out = model.conv_block3(out)
    print(out.shape)
    print(out.view(1, -1).shape[1])

torch.Size([1, 128, 16, 16])
32768


In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

EPOCHS = 30
train_losses = []
test_accuracies = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    test_accuracies.append(acc)

    print(f'Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

Epoch [1/30] Loss: 7.6341 | Test Acc: 0.2014
Epoch [2/30] Loss: 1.9079 | Test Acc: 0.3125
Epoch [3/30] Loss: 1.8812 | Test Acc: 0.2604
Epoch [4/30] Loss: 1.8439 | Test Acc: 0.3229
Epoch [5/30] Loss: 1.8038 | Test Acc: 0.3299
Epoch [6/30] Loss: 1.7832 | Test Acc: 0.3264
Epoch [7/30] Loss: 1.7904 | Test Acc: 0.3715
Epoch [8/30] Loss: 1.7502 | Test Acc: 0.2882
Epoch [9/30] Loss: 1.7304 | Test Acc: 0.3194
Epoch [10/30] Loss: 1.7653 | Test Acc: 0.3438
Epoch [11/30] Loss: 1.7022 | Test Acc: 0.3438
Epoch [12/30] Loss: 1.6672 | Test Acc: 0.3368
Epoch [13/30] Loss: 1.6296 | Test Acc: 0.3750
Epoch [14/30] Loss: 1.5955 | Test Acc: 0.3646
Epoch [15/30] Loss: 1.5769 | Test Acc: 0.3958
Epoch [16/30] Loss: 1.5651 | Test Acc: 0.3958
Epoch [17/30] Loss: 1.5061 | Test Acc: 0.4306
Epoch [18/30] Loss: 1.5421 | Test Acc: 0.4271
Epoch [19/30] Loss: 1.5213 | Test Acc: 0.4340
Epoch [20/30] Loss: 1.4779 | Test Acc: 0.4062
Epoch [21/30] Loss: 1.4075 | Test Acc: 0.4444
Epoch [22/30] Loss: 1.3682 | Test Acc: 0.44

In [15]:
EPOCHS2 = 30

for epoch in range(EPOCHS2):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    test_accuracies.append(acc)

    print(f'Epoch [{epoch+31}/{60}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

Epoch [31/60] Loss: 1.1715 | Test Acc: 0.4653
Epoch [32/60] Loss: 1.1567 | Test Acc: 0.5000
Epoch [33/60] Loss: 1.1659 | Test Acc: 0.5139
Epoch [34/60] Loss: 1.1483 | Test Acc: 0.5243
Epoch [35/60] Loss: 1.1143 | Test Acc: 0.5174
Epoch [36/60] Loss: 1.0782 | Test Acc: 0.5139
Epoch [37/60] Loss: 1.0741 | Test Acc: 0.5104
Epoch [38/60] Loss: 1.0978 | Test Acc: 0.5035
Epoch [39/60] Loss: 1.0505 | Test Acc: 0.5382
Epoch [40/60] Loss: 1.0411 | Test Acc: 0.5104
Epoch [41/60] Loss: 1.0288 | Test Acc: 0.5278
Epoch [42/60] Loss: 1.0286 | Test Acc: 0.5312
Epoch [43/60] Loss: 1.0071 | Test Acc: 0.5174
Epoch [44/60] Loss: 0.9535 | Test Acc: 0.5312
Epoch [45/60] Loss: 0.9699 | Test Acc: 0.5174
Epoch [46/60] Loss: 0.9471 | Test Acc: 0.5278
Epoch [47/60] Loss: 0.9726 | Test Acc: 0.5139
Epoch [48/60] Loss: 0.9565 | Test Acc: 0.5208
Epoch [49/60] Loss: 0.9381 | Test Acc: 0.5660
Epoch [50/60] Loss: 0.9238 | Test Acc: 0.5451
Epoch [51/60] Loss: 0.8852 | Test Acc: 0.5382
Epoch [52/60] Loss: 0.9020 | Test 

In [16]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import torchvision.models as models
efficientnet = models.efficientnet_b0(weights='IMAGENET1K_V1')
print(efficientnet.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [17]:
# Replace final layer for 8 classes
efficientnet.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(1280, NUM_CLASSES)
)

# Convert grayscale (1 channel) to 3 channels EfficientNet expects
efficientnet.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)

# Move to GPU
efficientnet = efficientnet.to(device)

# Freeze all layers except classifier
for param in efficientnet.parameters():
    param.requires_grad = False

# Unfreeze classifier only
for param in efficientnet.classifier.parameters():
    param.requires_grad = True

# Unfreeze first conv too since we changed it
for param in efficientnet.features[0].parameters():
    param.requires_grad = True

print('EfficientNet ready')
print(f'Classifier: {efficientnet.classifier}')

EfficientNet ready
Classifier: Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=8, bias=True)
)


In [ ]:
optimizer_eff = optim.Adam(
    filter(lambda p: p.requires_grad, efficientnet.parameters()),
    lr=0.001
)
scheduler_eff = optim.lr_scheduler.StepLR(optimizer_eff, step_size=10, gamma=0.5)

EPOCHS_EFF = 30
eff_losses = []
eff_accuracies = []

for epoch in range(EPOCHS_EFF):
    efficientnet.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_eff.zero_grad()
        outputs = efficientnet(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer_eff.step()
        running_loss += loss.item()

    scheduler_eff.step()

    efficientnet.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = efficientnet(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    eff_losses.append(avg_loss)
    eff_accuracies.append(acc)

    print(f'Epoch [{epoch+1}/{EPOCHS_EFF}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

Epoch [1/30] Loss: 1.9945 | Test Acc: 0.1875
